In [12]:
import platform
import psutil
from typing import Tuple, Union
from timeit import timeit
from typing import Callable
from warnings import warn

# PyTorch dependencies
import torch
import torch.backends.opt_einsum as opt_einsum
from torch.func import jacfwd
from torch import Tensor


# Internal dependencies
from thoad import backward, Controller

In [13]:
# control size of tensors
TENSOR_SCALE: Union[int, float] = 1
REPEAT_SCALE: Union[int, float] = 1

In [14]:
sys: platform.uname_result = platform.uname()
print(f"system           {sys.system} {sys.release} {sys.version}")

system           Windows 11 10.0.22631


In [15]:
torch_dev: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch_dev.type == 'cuda':
    idx = torch_dev.index if torch_dev.index is not None else 0
    props: "_CudaDeviceProperties" = torch.cuda.get_device_properties(idx)
    name: str = props.name
    total_mem_gb: float = props.total_memory / (1024**3)
    print(f"using device     {torch_dev} -> {name}")
    print(f"device memory    {total_mem_gb:.1f} GB)")
else:
    cpu_name: str = platform.processor() or "CPU"
    print(f"using device     {torch_dev} -> {cpu_name}")
    print(f"physical cores   {psutil.cpu_count(logical=False)}")
    print(f"logical cores    {psutil.cpu_count(logical=True)}")

using device     cuda -> NVIDIA GeForce RTX 4070 Ti
device memory    12.0 GB)


In [16]:
if opt_einsum.is_available():
    opt_einsum.enabled = True
    print("opt_einsum backend enabled")
    opt_einsum.strategy = "optimal"
else:
    warn(
        "opt_einsum backend is not available. "
        "For better performance, install and enable opt_einsum.",
        UserWarning
    )

C:\Users\stic\AppData\Local\Temp\ipykernel_8920\428334837.py:6: UserWarning: opt_einsum backend is not available. For better performance, install and enable opt_einsum.
  warn(


## **Benchmark differentiations on full MLP**

definition of MLP

In [17]:
def torch_foward_pass(X: Tensor, *params) -> Tensor:
    T: Tensor = X
    for i, P in enumerate(params):
        last_step: bool = i == (len(params) - 1)
        T = T @ P
        T = torch.softmax(T, dim=1) if last_step else torch.relu(T)
    return T

definition of helper functions to meassure differentiation times

In [18]:
def compose_jacfwd(fn: Callable, n: int) -> Callable:
    g: Callable = fn
    for _ in range(n):
        g = jacfwd(g)
    return g


def time_func_differentiation(reps: int, order:int, X: Tensor, *params) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(False) for P in params]
    assert all(not param.requires_grad for param in params)
    def _fixed_forward(X: Tensor) -> Tensor:
        Y: Tensor = torch_foward_pass(X, *params)
        return Y
    differentiator: Callable = compose_jacfwd(fn=_fixed_forward, n=order)
    def _foward_and_backward() -> None:
        differentiator(X)
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time


def time_thoad_differentiation(reps: int, order:int, X: Tensor, *params) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(False) for P in params]
    assert all(not param.requires_grad for param in params)
    def _foward_and_backward() -> None:
        T: Tensor = torch_foward_pass(X, *params)
        ctrl: Controller = backward(
            tensor=T,
            order=order,
            crossings=False,
            keep_batch=True,
        )
        ctrl.clear()
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

differentiation computational cost w.r.t. **order** and **batch size**

In [19]:
for o in [1, 2, 3]:
    print(f"\nORDER {o}")
    for batch_size in [15, 30, 45, 60, 75, 90]:
        batch_size //= o
        param_size: int = int(5 * TENSOR_SCALE)
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        # create torch tensors
        torch_X: Tensor = torch.rand(size=x_shape, device=torch_dev)
        torch_params: list[Tensor] = [
            torch.rand(size=p_shape, device=torch_dev) for _ in range(3)
        ]

        reps: int = int(600 * (1 / batch_size) * (1/o) * REPEAT_SCALE)
        func_time: float = time_func_differentiation(reps, o, torch_X, *torch_params)
        thoad_time: float = time_thoad_differentiation(reps, o, torch_X, *torch_params)
        print(
            f"batch size: {batch_size:02d} -> "
            f"func time: {func_time/reps:.4f}  thoad time: {thoad_time/reps:.4f}"
        )


ORDER 1
batch size: 15 -> func time: 0.0011  thoad time: 0.0043
batch size: 30 -> func time: 0.0007  thoad time: 0.0044
batch size: 45 -> func time: 0.0007  thoad time: 0.0043
batch size: 60 -> func time: 0.0008  thoad time: 0.0044
batch size: 75 -> func time: 0.0010  thoad time: 0.0044
batch size: 90 -> func time: 0.0009  thoad time: 0.0039

ORDER 2
batch size: 07 -> func time: 0.0033  thoad time: 0.0073
batch size: 15 -> func time: 0.0031  thoad time: 0.0075
batch size: 22 -> func time: 0.0026  thoad time: 0.0071
batch size: 30 -> func time: 0.0092  thoad time: 0.0093
batch size: 37 -> func time: 0.0093  thoad time: 0.0080
batch size: 45 -> func time: 0.0032  thoad time: 0.0070

ORDER 3
batch size: 05 -> func time: 0.0110  thoad time: 0.0107
batch size: 10 -> func time: 0.0094  thoad time: 0.0111
batch size: 15 -> func time: 0.0196  thoad time: 0.0116
batch size: 20 -> func time: 0.0597  thoad time: 0.0166
batch size: 25 -> func time: 0.1348  thoad time: 0.0253
batch size: 30 -> fun

differentiation computational cost w.r.t. **order** and **param size**

In [20]:
for o in [1, 2, 3]:
    print(f"\nORDER {o}")
    for param_size in [10, 20, 30, 40, 50, 60]:
        batch_size: int = int(5 * TENSOR_SCALE)
        param_size //= o
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        # create torch tensors
        torch_X: Tensor = torch.rand(size=x_shape, device=torch_dev)
        torch_params: list[Tensor] = [
            torch.rand(size=p_shape, device=torch_dev) for _ in range(3)
        ]

        reps: int = int(600 * (1 / param_size) * (1/o) * REPEAT_SCALE)
        func_time: float = time_func_differentiation(reps, o, torch_X, *torch_params)
        thoad_time: float = time_thoad_differentiation(reps, o, torch_X, *torch_params)
        print(
            f"param size: {param_size:02d} -> "
            f"func time: {func_time/reps:.4f}  thoad time: {thoad_time/reps:.4f}"
        )




ORDER 1
param size: 10 -> func time: 0.0010  thoad time: 0.0046
param size: 20 -> func time: 0.0009  thoad time: 0.0046
param size: 30 -> func time: 0.0009  thoad time: 0.0043
param size: 40 -> func time: 0.0007  thoad time: 0.0042
param size: 50 -> func time: 0.0008  thoad time: 0.0044
param size: 60 -> func time: 0.0008  thoad time: 0.0041

ORDER 2
param size: 05 -> func time: 0.0033  thoad time: 0.0073
param size: 10 -> func time: 0.0034  thoad time: 0.0075
param size: 15 -> func time: 0.0032  thoad time: 0.0082
param size: 20 -> func time: 0.0038  thoad time: 0.0075
param size: 25 -> func time: 0.0034  thoad time: 0.0077
param size: 30 -> func time: 0.0063  thoad time: 0.0075

ORDER 3
param size: 03 -> func time: 0.0103  thoad time: 0.0108
param size: 06 -> func time: 0.0103  thoad time: 0.0106
param size: 10 -> func time: 0.0221  thoad time: 0.0107
param size: 13 -> func time: 0.0130  thoad time: 0.0105
param size: 16 -> func time: 0.0265  thoad time: 0.0134
param size: 20 -> fun

differentiation computational cost w.r.t. **graph depth** (param gradients included)

In [21]:
for o in [1, 2, 3]:
    print(f"\nORDER {o}")
    for depth in [2, 3, 4, 5, 6, 7, 8]:
        batch_size: int = 40 // o
        param_size: int = int(10 // o * TENSOR_SCALE)
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        # create torch tensors
        torch_X: Tensor = torch.rand(size=x_shape, device=torch_dev)
        torch_params: list[Tensor] = [
            torch.rand(size=p_shape, device=torch_dev) for _ in range(depth)
        ]

        reps: int = int(600 * (1 / depth) * (1/o) * REPEAT_SCALE)
        func_time: float = time_func_differentiation(reps, o, torch_X, *torch_params)
        thoad_time: float = time_thoad_differentiation(reps, o, torch_X, *torch_params)
        print(
            f"depth size: {depth:02d} -> "
            f"func time: {func_time/reps:.4f}  thoad time: {thoad_time/reps:.4f}"
        )


ORDER 1
depth size: 02 -> func time: 0.0008  thoad time: 0.0035
depth size: 03 -> func time: 0.0009  thoad time: 0.0045
depth size: 04 -> func time: 0.0012  thoad time: 0.0053
depth size: 05 -> func time: 0.0016  thoad time: 0.0063
depth size: 06 -> func time: 0.0016  thoad time: 0.0076
depth size: 07 -> func time: 0.0017  thoad time: 0.0080
depth size: 08 -> func time: 0.0016  thoad time: 0.0089

ORDER 2
depth size: 02 -> func time: 0.0025  thoad time: 0.0058
depth size: 03 -> func time: 0.0031  thoad time: 0.0073
depth size: 04 -> func time: 0.0041  thoad time: 0.0090
depth size: 05 -> func time: 0.0042  thoad time: 0.0109
depth size: 06 -> func time: 0.0051  thoad time: 0.0124
depth size: 07 -> func time: 0.0054  thoad time: 0.0149
depth size: 08 -> func time: 0.0065  thoad time: 0.0164

ORDER 3
depth size: 02 -> func time: 0.0082  thoad time: 0.0080
depth size: 03 -> func time: 0.0095  thoad time: 0.0105
depth size: 04 -> func time: 0.0122  thoad time: 0.0131
depth size: 05 -> fun